## Libraries

In [ ]:
from neo4j_config import get_driver, NEO4J_DATABASE
from neo4j import GraphDatabase
import pandas as pd

## Session

In [ ]:
driver = get_driver()

In [4]:
driver.execute_query("SHOW DATABASES yield name")

EagerResult(records=[<Record name='neo4j'>, <Record name='system'>], summary=<neo4j._work.summary.ResultSummary object at 0x00000174EB28D510>, keys=['name'])

## Criando o Banco de Grafos

Constraints

In [9]:
def criar_constraints():
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE")
        session.run("CREATE CONSTRAINT track_id IF NOT EXISTS FOR (t:Track) REQUIRE t.track_id IS UNIQUE")
        session.run("CREATE CONSTRAINT artist_id IF NOT EXISTS FOR (a:Artist) REQUIRE a.artist_id IS UNIQUE")
        session.run("CREATE CONSTRAINT genre_id IF NOT EXISTS FOR (g:Genre) REQUIRE g.genre_id IS UNIQUE")

criar_constraints()

Track Nodes

In [14]:
def track_nodes():
    query = """
LOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1Ak45D5Cx5ifWpYcVql2FpxWoXKZ80WDb' AS row
CALL (row) {
    MERGE (t:Track {track_id: row.track_id})
    SET t += {
        name: row.name,
        year: toInteger(row.year),
        danceability: toFloat(row.danceability),
        energy: toFloat(row.energy)
    }
} IN TRANSACTIONS OF 1000 rows
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

track_nodes()


Artist Nodes

In [16]:
def artist_nodes():
    query = """
LOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1Ak45D5Cx5ifWpYcVql2FpxWoXKZ80WDb' AS row
CALL (row) {
    MATCH (t:Track {track_id: row.track_id})
    MERGE (a:Artist {artist_id: row.artist})
    SET a.name = row.artist
    MERGE (t)-[: BY_ARTIST]->(a)
} IN TRANSACTIONS OF 1000 rows
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

artist_nodes()

Genres Nodes

In [20]:
def genre_nodes():
    query = """
LOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1Ak45D5Cx5ifWpYcVql2FpxWoXKZ80WDb' AS row
WITH row WHERE row.genre IS NOT NULL AND row.genre <> ''
CALL {
  WITH row
  MATCH (t:Track {track_id: row.track_id})
  WITH row, t, split(row.genre, ',') AS genres
  UNWIND genres AS g
  WITH t, trim(g) AS genre WHERE genre <> ''
  MERGE (g:Genre {genre_id: genre})
  SET g.type = genre
  MERGE (t)-[:HAS_GENRE]->(g)
} IN TRANSACTIONS OF 1000 ROWS;
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

genre_nodes()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=4, column=1, offset=175>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 175, 'line': 4, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nLOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1Ak45D5Cx5ifWpYcVql2FpxWoXKZ80WDb' AS row\nWITH row WHERE row.genre IS NOT NULL AND row.genre <> ''\nCALL {\n  WITH row\n  MATCH (t:Track {track_id: row.track_id})\n  WITH row, t, split(row.genre, ',') AS genres\n  UNWIND genres AS g\n  WITH t, trim(g) AS genre W

User Nodes

In [21]:
def user_nodes():
    query = """
LOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=13D081hvax77-9YOONXwoB7v87fE9_s2Z' AS row
CALL (row) {
    MERGE (u:User {user_id: row.user_id})
    SET u += {
        username: row.username,
        age: toInteger(row.age),
        sex: row.sex,
        country: row.country,
        subscription_type: row.subscription_type,
        signup_date: datetime(row.signup_date),
        last_active: datetime(row.last_active)
    }
} IN TRANSACTIONS OF 1000 rows
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

user_nodes()

LISTENED_TO Relationship

In [28]:
def listened_to():
    query = """
LOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1ZTbwLuEr9KR4AkFk6ttfe89V7lWw5auW' AS row
WITH row.user_id AS user, row.track_id AS track, toInteger(row.playcount) AS pc
ORDER BY user, pc DESC
WITH user, collect({track: track, pc: pc})[0..5] AS top    
UNWIND top AS item
CALL {
  WITH user, item
  MATCH (u:User {user_id: user})
  MATCH (t:Track {track_id: item.track})
  MERGE (u)-[r:LISTENED_TO]->(t)
  SET r.playcount = item.pc
} IN TRANSACTIONS OF 1000 ROWS;
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

listened_to()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (user, item) { ... }', position=<SummaryInputPosition line=7, column=1, offset=300>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 300, 'line': 7, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nLOAD CSV WITH HEADERS FROM 'https://drive.google.com/uc?export=download&id=1ZTbwLuEr9KR4AkFk6ttfe89V7lWw5auW' AS row\nWITH row.user_id AS user, row.track_id AS track, toInteger(row.playcount) AS pc\nORDER BY user, pc DESC\nWITH user, collect({track: track, pc: pc})[0..5] AS top    \nUNWIND top AS item\nCALL {\n  WITH user, item\n

## Calculated Attributes

Artist Attributes

In [37]:
def artist_attrs():
    query = """
MATCH (a:Artist)<-[:BY_ARTIST]-(t:Track)<-[r:LISTENED_TO]-(:User)
WITH a, sum(r.playcount) AS total_plays
SET a.total_playcounts = total_plays;
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

artist_attrs()
    

Track Attributes

In [38]:
def track_attrs():
    query = """
MATCH (t:Track)<-[r:LISTENED_TO]-(u:User)
WITH t, 
     sum(r.playcount) AS total_plays,
     count(DISTINCT u) AS listeners
SET t.total_playcount  = total_plays,
    t.listeners_count  = listeners;
"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query)

track_attrs()

User Attributes

In [39]:
def user_attrs(tx):
    tx.run("""
        MATCH (u:User)-[r:LISTENED_TO]->(t:Track)
        WITH u,
             sum(r.playcount) AS total_plays,
             count(DISTINCT t) AS unique_tracks
        SET u.total_plays   = total_plays,
            u.unique_tracks_cnt = unique_tracks
    """)

    tx.run("""
        MATCH (u:User)-[r:LISTENED_TO]->(t:Track)-[:BY_ARTIST]->(a:Artist)
        WITH u, a, sum(r.playcount) AS plays
        ORDER BY plays DESC
        WITH u, collect({artist_id: a.artist_id, plays: plays}) AS artists
        WITH u, head(artists) AS fav
        WHERE fav IS NOT NULL
        SET u.favorite_artist = fav.artist_id
    """)

    tx.run("""
        MATCH (u:User)-[r:LISTENED_TO]->(t:Track)
        WITH u,
            avg(t.danceability) AS avg_dance,
            avg(t.energy) AS avg_energy
        SET u += {
           avg_danceability: avg_dance,
           avg_energy: avg_energy
        }
    """)

with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(user_attrs)


## Music List Recommendation

Sugere para cada usuário faixas de artistas que ele já ouviu, mas que ele ainda não escutou, ranqueando pela soma dos playcounts de outros ouvintes.

In [60]:
query = """
MATCH (u:User)
WITH u LIMIT 100
MATCH (u)-[:LISTENED_TO]->(:Track)-[:BY_ARTIST]->(a:Artist)
MATCH (a)<-[:BY_ARTIST]-(rec:Track)<-[r:LISTENED_TO]-(other:User)
WHERE u <> other AND NOT (u)-[:LISTENED_TO]->(rec)
WITH u, rec, sum(r.playcount) AS score
ORDER BY u.user_id, score DESC
WITH u, collect({track: rec.name, artist: rec.artist, score: score})[0..3] AS recs
RETURN u.username AS user, recs;
"""

with driver.session(database=NEO4J_DATABASE) as session:
    records = session.run(query).data()

# montar DataFrame
df = pd.DataFrame(records)
display(df.head())

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `artist` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=9, column=47, offset=326>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 326, 'line': 9, 'column': 47}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (u:User)\nWITH u LIMIT 100\nMATCH (u)-[:LISTENED_TO]->(:Track)-[:BY_ARTIST]->(a:Artist)\nMATCH (a)<-[:BY_ARTIST]-(rec:Track)<-[r:LISTENED_TO]-(other:User)\nWHERE u <> other AND NOT (u)-[:LISTENED_TO]->(rec)\nWITH u, rec, sum(r.playcount) AS score\nORDER BY u.user_id, score DESC\nWITH u, collect({track: re

,user,recs
0,daniela.pereira0047,"[{'score': 6824, 'artist': None, 'track': 'On ..."
1,carla.alves0087,"[{'score': 3416, 'artist': None, 'track': 'Pre..."
2,john.souza0085,"[{'score': 16620, 'artist': None, 'track': 'Ev..."
3,juliana.brown0067,"[{'score': 2107, 'artist': None, 'track': 'The..."
4,daniela.souza0017,"[{'score': 10350, 'artist': None, 'track': 'Ta..."
